In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F

# gjej range real nga orders (kerkesa do full range of the data)
bounds = spark.table(f"{catalog_name}.{silver_schema}.orders").select(
    F.min("order_purchase_timestamp").alias("min_d"),
    F.max("order_purchase_timestamp").alias("max_d")
).collect()[0]

start_date = bounds["min_d"].date()
end_date = bounds["max_d"].date()
print(f"Date range: {start_date} -> {end_date}")

# gjenero nje record per cdo dite (sekuence ben listen e datave, explode e kthen ne recorde)
date_df = (spark.sql(f"SELECT sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day) as dates")
    .withColumn("date", F.explode("dates")).drop("dates"))

dim_date = (date_df
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7)))

(dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_date"))
print(f"Wrote {dim_date.count():,} dates")